In [14]:
import pandas as pd
import numpy as np
import joblib
import os
import xgboost as xgb
import lightgbm as lgb
import catboost as cb
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# ✅ 1. 파일 로드
train_path = "/root/Public_Storage/madelab_khw/lg_aimers/dataset/train.csv"
df = pd.read_csv(train_path)

# ✅ 2. 데이터 전처리 함수
def pre_process(df, is_train=True):
    df = df.drop(columns=['ID'], errors='ignore')  # ✅ ID 제거
    if '임신 성공 여부' in df.columns:
        df_sol = df.pop('임신 성공 여부')  # ✅ y(Label) 분리
    else:
        df_sol = None

    for col in df.columns:
        if df[col].dtype == 'object':
            df[col].fillna('Unknown', inplace=True)
        else:
            if df[col].nunique()<=4:
                df[col].fillna(0, inplace = True)
            else:
                df[col].fillna(df[col].mean(), inplace=True)


    # ✅ Label Encoding
    categorical_columns = df.select_dtypes(include=['object']).columns.tolist()
    if is_train:
        encoders = {col: LabelEncoder().fit(df[col].astype(str)) for col in categorical_columns}
        joblib.dump(encoders, "encoders.pkl")  # ✅ 인코더 저장
    else:
        encoders = joblib.load("encoders.pkl")  # ✅ 인코더 로드

    for col in categorical_columns:
        df[col] = encoders[col].transform(df[col].astype(str))

    # ✅ MinMax Scaling
    numeric_columns = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
    if is_train:
        scaler = MinMaxScaler()
        df[numeric_columns] = scaler.fit_transform(df[numeric_columns])
        joblib.dump(scaler, "scaler.pkl")  # ✅ 스케일러 저장
    else:
        scaler = joblib.load("scaler.pkl")
        df[numeric_columns] = scaler.transform(df[numeric_columns])

    return df, df_sol  # ✅ X, y 반환

# ✅ 데이터 전처리 수행
X, y = pre_process(df, is_train=True)

# ✅ Train/Validation 데이터 분할
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ✅ Train 컬럼 순서 저장
joblib.dump(X_train.columns.tolist(), "train_columns.pkl")

# ✅ 특정 컬럼 가중치 설정 (XGBoost, LightGBM, CatBoost에 적용)
weight_columns = ['시술 당시 나이', '이식된 배아 수']
weights = np.ones(X_train.shape[0])  # 기본 가중치 1.0
for col in weight_columns:
    if col in X_train.columns:
        weights += X_train[col].values * 1.2  # **가중치 추가**

# ✅ 3. 모델 학습 (XGBoost, LightGBM, CatBoost)

# ✅ XGBoost 학습
params_xgb = {
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "learning_rate": 0.0055,
    "max_depth": 6,
    "subsample": 0.85,
    "colsample_bytree": 0.75,
    "min_child_weight": 3,
    "lambda": 1.0,
    "alpha": 0.5,
    "gamma": 0.3,
    "scale_pos_weight": 2.1,
    "random_state": 42,
}
dtrain = xgb.DMatrix(X_train, label=y_train)
dtrain.set_weight(weights)  # **가중치 적용**
dvalid = xgb.DMatrix(X_valid, label=y_valid)

model_xgb = xgb.train(params_xgb, dtrain, num_boost_round=2000, evals=[(dvalid, "valid")], early_stopping_rounds=200, verbose_eval=10)

# ✅ LightGBM 학습
params_lgb = {
    "objective": "binary",
    "metric": "auc",
    "learning_rate": 0.0055,
    "num_leaves": 45,
    "max_depth": 7,
    "subsample": 0.87,
    "colsample_bytree": 0.90,
    "reg_alpha": 0.4,
    "reg_lambda": 0.8,
    "random_state": 42,
    "scale_pos_weight": 2.0,
}
lgb_train = lgb.Dataset(X_train, label=y_train, weight=weights)  # **가중치 적용**
lgb_valid = lgb.Dataset(X_valid, label=y_valid)
model_lgb = lgb.train(params_lgb, lgb_train, num_boost_round=1800, valid_sets=[lgb_valid], callbacks=[lgb.callback.early_stopping(200), lgb.callback.log_evaluation(20)])

# ✅ CatBoost 학습
model_cat = cb.CatBoostClassifier(iterations=1200, learning_rate=0.015, depth=6, random_seed=42, class_weights=[1, 2])
model_cat.fit(X_train, y_train, sample_weight=weights, eval_set=(X_valid, y_valid), verbose=100)  # **가중치 적용**

# ✅ 4. Soft Voting 앙상블 (30:30:30 비율)
test_preds = np.zeros((X_valid.shape[0], 3))

test_preds[:, 0] = model_xgb.predict(xgb.DMatrix(X_valid))  # XGBoost 예측
test_preds[:, 1] = model_lgb.predict(X_valid)  # LightGBM 예측
test_preds[:, 2] = model_cat.predict_proba(X_valid)[:, 1]  # CatBoost 예측

final_test_preds = (
    0.30 * test_preds[:, 0] +
    0.30 * test_preds[:, 1] +
    0.30 * test_preds[:, 2]
)

# ✅ 5. 최종 평가
ensemble_auc = roc_auc_score(y_valid, final_test_preds)
print(f"🚀 Soft Voting 앙상블 ROC-AUC: {ensemble_auc:.4f}")

# ✅ 모델 저장
model_xgb.save_model("3xgb_model.json")
model_lgb.save_model("3lgb_model.txt")
model_cat.save_model("3cat_model.cbm")

print("✅ 모든 모델 저장 완료!")


[0]	valid-auc:0.72134
[10]	valid-auc:0.72846
[20]	valid-auc:0.72881
[30]	valid-auc:0.72931
[40]	valid-auc:0.72975
[50]	valid-auc:0.72990
[60]	valid-auc:0.73002
[70]	valid-auc:0.73025
[80]	valid-auc:0.73031
[90]	valid-auc:0.73038
[100]	valid-auc:0.73048
[110]	valid-auc:0.73071
[120]	valid-auc:0.73089
[130]	valid-auc:0.73101
[140]	valid-auc:0.73112
[150]	valid-auc:0.73130
[160]	valid-auc:0.73144
[170]	valid-auc:0.73157
[180]	valid-auc:0.73170
[190]	valid-auc:0.73183
[200]	valid-auc:0.73197
[210]	valid-auc:0.73207
[220]	valid-auc:0.73219
[230]	valid-auc:0.73234
[240]	valid-auc:0.73249
[250]	valid-auc:0.73261
[260]	valid-auc:0.73275
[270]	valid-auc:0.73286
[280]	valid-auc:0.73296
[290]	valid-auc:0.73305
[300]	valid-auc:0.73315
[310]	valid-auc:0.73326
[320]	valid-auc:0.73335
[330]	valid-auc:0.73345
[340]	valid-auc:0.73356
[350]	valid-auc:0.73368
[360]	valid-auc:0.73376
[370]	valid-auc:0.73385
[380]	valid-auc:0.73394
[390]	valid-auc:0.73403
[400]	valid-auc:0.73411
[410]	valid-auc:0.73421
[42